In [ ]:
%matplotlib widget
import sys
import time
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
import struct

def time_axis(scope, sig, unit='us'):
    la_rate = scope.LA.sampling_clock_frequency  # Hz; verify this name against print(scope.LA)
    sample_period = 1.0 / la_rate                # seconds per LA sample
    scale = {'s': 1, 'ms': 1e3, 'us': 1e6, 'ns': 1e9}[unit]
    return np.arange(len(sig)) * sample_period * scale


def auto_unit(duration_s):
    """Pick a sensible time unit based on total duration in seconds."""
    if duration_s >= 1:
        return 's', 1
    elif duration_s >= 1e-3:
        return 'ms', 1e3
    elif duration_s >= 1e-6:
        return 'us', 1e6
    else:
        return 'ns', 1e9

def plot_la_trace_glitch(scope, sigs):
    # diagnostic, same as before — check this before trusting the plot
    if debug:
        print("signals extracted:", list(sigs.keys()))
        for i, s in sigs.items():
            print(f"bit {i}: len={len(s)} min={min(s)} max={max(s)} transitions={sum(abs(s[1:] - s[:-1]))}")
        print("LA errors:", scope.errors)

    glitch_labels = {
        0: "glitch out",
        1: "source clock",
        2: "MMCM1",
        3: "MMCM2",
        4: "glitch go",
        5: "capture trigger",
        6: "glitch enable",
        7: "manual trigger",
        8: "MMCM1 trigger",
    }
    
    fig, axes = plt.subplots(len(sigs), 1, figsize=(12, 1.2 * len(sigs)), sharex=True, squeeze=False)
    for ax, (i, s) in zip(axes[:, 0], sigs.items()):
        t = time_axis(scope, s, unit)
        ax.step(t, s, where="post")     # t must be the x argument here
        ax.set_ylabel(glitch_labels.get(i, f"bit {i}"))
        ax.set_yticks([0, 1])
        ax.set_ylim(-0.2, 1.2)
    axes[-1, 0].set_xlabel(f"time ({unit})")
    plt.tight_layout()
    plt.show()


def plot_power_trace(trace, title="Power trace", unit=None):
    if trace is None:
        print("No trace to plot")
        return
    adc_rate = scope.clock.adc_freq  # confirm attribute name against print(scope.clock)
    n = len(trace)
    duration_s = n / adc_rate

    if unit is None:
        unit, scale = auto_unit(duration_s)
    else:
        scale = {'s': 1, 'ms': 1e3, 'us': 1e6, 'ns': 1e9}[unit]

    t = (np.arange(n) / adc_rate) * scale

    plt.figure(figsize=(12, 4))
    plt.plot(t, trace, linewidth=0.8)
    plt.xlabel(f"time ({unit})")
    plt.ylabel("ADC value")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_la_trace_cw_20p(scope, sigs):
    labels = {0: "tio1 (TX)", 1: "tio2 (RX)", 2: "tio3"}
    fig, axes = plt.subplots(len(labels), 1, figsize=(12, 3), sharex=True, squeeze=False)
    for ax, (i, name) in zip(axes[:, 0], labels.items()):
        sig = sigs[i]
        t = time_axis(scope,sig, unit)
        ax.step(t, sig, where="post")   # t must be the x argument here
        ax.set_ylabel(name)
        ax.set_yticks([0, 1])
        ax.set_ylim(-0.2, 1.2)
    axes[-1, 0].set_xlabel(f"time ({unit})")
    plt.tight_layout()
    plt.show()


def plot_la_trace_cw_20p_interactive(scope, sigs, unit="us", snap=True):
    labels = {0: "tio1 (TX)", 1: "tio2 (RX)", 2: "tio3"}
    fig, axes = plt.subplots(len(labels), 1, figsize=(12, 6),
                             sharex=True, squeeze=False)
    axs = axes[:, 0]

    edge_times = []
    for ax, (i, name) in zip(axs, labels.items()):
        sig = np.asarray(sigs[i])
        t = np.asarray(time_axis(scope, sig, unit))
        ax.step(t, sig, where="post")
        ax.set_ylabel(name)
        ax.set_yticks([0, 1])
        ax.set_ylim(-0.2, 1.2)
        ax.grid(True, alpha=0.3)
        # remember where the signal changes, for snapping
        edge_times.append(t[np.flatnonzero(np.diff(sig) != 0) + 1])

    edges = np.sort(np.concatenate(edge_times)) if edge_times else np.array([])
    axs[-1].set_xlabel(f"time ({unit})")
    axs[0].set_title("Left-click: place cursor A/B | Right-click: clear")

    marks = []   # x positions of the cursors
    lines = []   # the drawn vertical lines

    def clear():
        for ln in lines:
            ln.remove()
        lines.clear()
        marks.clear()

    def on_click(event):
        # ignore clicks while zoom/pan tool is active
        tb = fig.canvas.toolbar
        if tb is not None and getattr(tb, "mode", "") != "":
            return
        if event.inaxes not in axs:
            return

        if event.button == 3:          # right-click clears
            clear()
            axs[0].set_title("Left-click: place cursor A/B | Right-click: clear")
            fig.canvas.draw_idle()
            return
        if event.button != 1:
            return

        x = event.xdata
        if snap and edges.size:
            x = edges[np.argmin(np.abs(edges - x))]

        if len(marks) == 2:            # start a new measurement
            clear()

        marks.append(x)
        color = "tab:red" if len(marks) == 1 else "tab:green"
        for ax in axs:
            lines.append(ax.axvline(x, color=color, linestyle="--", lw=1))

        if len(marks) == 1:
            axs[0].set_title(f"A = {x:.6g} {unit}")
        else:
            dt = abs(marks[1] - marks[0])
            freq = f"{1 / dt:.6g} 1/{unit}" if dt > 0 else "n/a"
            axs[0].set_title(
                f"A = {marks[0]:.6g} {unit}   B = {marks[1]:.6g} {unit}   "
                f"Δt = {dt:.6g} {unit}   (1/Δt = {freq})"
            )
        fig.canvas.draw_idle()

    plt.ioff()
    fig.canvas.mpl_connect("button_press_event", on_click)
    plt.tight_layout()
    plt.show()
    return fig  # keep a reference so the callback isn't garbage-collected
    